In [ ]:
# !pip install openpyxl

In [ ]:
## Packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import f as f_dist
from adjustText import adjust_text

In [ ]:
## Rechargement des données nettoyées (issues du fichier 01)
directory_data = '../'
# Chemin de sortie
SORTIE = "../Travail Julie/"
df = pd.read_csv(SORTIE + 'engie_data_clean.csv', sep=';', decimal='.')

In [ ]:
## ---------------------------------------------------------
## Définition centralisée des features / cible
## ---------------------------------------------------------
cols_to_exclude = ['TARGET', 'MAC_CODE', 'Date_time', 'ID']
features = list(df.columns.difference(cols_to_exclude))

X = df[features]
y = df['TARGET']

In [ ]:
## Description de la base
print("--- Aperçu des premières lignes du DataFrame ---")
display(df.head())

print("--- Aperçu des premières lignes de la variable TARGET  ---")
display(df['TARGET'].head())

print("\n--- Informations générales sur le DataFrame (types, non-null) ---")
df.info()

print("\n--- Statistiques descriptives ---")
display(df.describe())

print("\n--- Dimensions du DataFrame (lignes, colonnes) ---")
print(df.shape)

print("\n--- Types de données des colonnes ---")
display(df.dtypes)

In [ ]:
## Focus valeurs manquantes
sns.heatmap(df.isnull(), cbar=False)
plt.title('Valeurs manquantes')
plt.show()

missing_percentage = df.isna().mean() * 100
print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

In [ ]:
### Analyse descriptive univariée

In [ ]:
## Boxplots - vue d'ensemble des variables numériques
numerical_features = X.select_dtypes(include=np.number).columns.tolist()

n_cols = 5
n_rows = (len(numerical_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3))
axes = axes.flatten()

print("--- Boxplots pour toutes les colonnes numériques (vue d'ensemble) ---")
for i, col in enumerate(numerical_features):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('')
    axes[i].tick_params(axis='y', labelsize=7)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

for j in range(len(numerical_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
## Scatter plots 1D - variables quantitatives colorées par valeurs manquantes ailleurs
n_cols = 4
n_rows = (len(numerical_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.5, n_rows * 3.5))
axes = axes.flatten()

print("--- Scatter plots 1D pour les variables quantitatives (points colorés par valeurs manquantes dans d'autres variables) ---")

for i, col in enumerate(numerical_features):
    ax = axes[i]
    cols_to_check_for_missing = [c for c in df.columns if c not in [col] + cols_to_exclude]
    has_other_missing = df.loc[df[col].notna(), cols_to_check_for_missing].isnull().any(axis=1)
    hue_labels = has_other_missing.map({True: 'Manquants dans autres vars', False: 'Pas de manquants ailleurs'})

    plot_data = pd.DataFrame({
        'value': df.loc[df[col].notna(), col],
        'has_missing': hue_labels
    })

    sns.stripplot(y='value', x=np.zeros(len(plot_data)), data=plot_data, hue='has_missing',
                  jitter=0.2, ax=ax, palette={'Manquants dans autres vars': 'red', 'Pas de manquants ailleurs': 'blue'},
                  alpha=0.4, legend=False)

    ax.set_title(col, fontsize=9)
    ax.set_ylabel('')
    ax.set_xlabel('')
    ax.set_xticks([])
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    if i == 0:
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, title="État des manquants", bbox_to_anchor=(1.02, 0.95), loc='upper left', borderaxespad=0.)

for j in range(len(numerical_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
### Analyse exploratoire bivariée

In [ ]:
## ---------------------------------------------------------
## 1. Matrice de corrélation (Pearson) entre variables numériques
## ---------------------------------------------------------
corr_matrix = X[numerical_features].assign(TARGET=y).corr(method='pearson')

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation (Pearson)")
plt.tight_layout()
plt.show()

# Option : version Spearman pour capter les relations non linéaires / monotones
corr_spearman = X[numerical_features].assign(TARGET=y).corr(method='spearman')
plt.figure(figsize=(12, 10))
sns.heatmap(corr_spearman, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation (Spearman)")
plt.tight_layout()
plt.show()


In [ ]:
## ---------------------------------------------------------
## 2. Corrélation de chaque variable avec la cible (TARGET), triée
## ---------------------------------------------------------
corr_with_target = corr_matrix['TARGET'].drop('TARGET').sort_values(key=abs, ascending=False)

print("--- Corrélation de chaque variable avec la cible (triée par force de la relation) ---")
print(corr_with_target)

plt.figure(figsize=(8, 6))
ax = sns.barplot(x=corr_with_target.values, y=corr_with_target.index, palette="coolwarm")
plt.title("Corrélation avec TARGET")
plt.axvline(0, color='black', linewidth=0.8)
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
## ---------------------------------------------------------
## 3. Scatter plots des variables les plus corrélées avec la cible
## ---------------------------------------------------------
top_n = 6
top_vars = corr_with_target.abs().sort_values(ascending=False).head(top_n).index

n_cols = 3
n_rows = (len(top_vars) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(top_vars):
    sns.scatterplot(x=X[col], y=y, ax=axes[i], alpha=0.4)
    axes[i].set_title(f"{col} vs TARGET")

for j in range(len(top_vars), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
## ---------------------------------------------------------
## 4. Détection de multi-colinéarité : VIF (Variance Inflation Factor)
## ---------------------------------------------------------
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# On regarde uniquement les features numériques (pas la cible)
X_vif_input = X[numerical_features].dropna()
X_vif = add_constant(X_vif_input)

vif_data = pd.DataFrame()
vif_data["variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("--- VIF (Variance Inflation Factor) par variable ---")
print(vif_data[vif_data['variable'] != 'const'].sort_values('VIF', ascending=False))
# Règle usuelle : VIF > 5 (voire > 10) => forte multi-colinéarité, variable candidate à retirer ou transformer

In [ ]:
## ---------------------------------------------------------
## 5. PCA pour visualiser la structure globale des observations
## ---------------------------------------------------------

df_pca_input = X[numerical_features].dropna()       # on conserve les variables numériques explicatives, en retirant les lignes avec valeurs manquantes

X_scaled = StandardScaler().fit_transform(df_pca_input)     # centrage/réduction
pca = PCA(n_components=3)
components = pca.fit_transform(X_scaled)

coord_vars = pca.components_.T * np.sqrt(pca.explained_variance_)
coord_df = pd.DataFrame(coord_vars, columns=['PC1', 'PC2', 'PC3'], index=numerical_features)

cos2_df = coord_df ** 2
cos2_df['cos2_total_3D'] = cos2_df[['PC1', 'PC2', 'PC3']].sum(axis=1)

contrib_df = pd.DataFrame((pca.components_.T ** 2) * 100,
                           columns=['PC1', 'PC2', 'PC3'], index=numerical_features)

print("Variance expliquée : PC1={:.1f}%, PC2={:.1f}%, PC3={:.1f}%".format(
    *(pca.explained_variance_ratio_[:3] * 100)))

# Biplot des 2 premières composantes
plt.figure(figsize=(8, 6))
sc = plt.scatter(components[:, 0], components[:, 1],
                  c=y.loc[df_pca_input.index], cmap='coolwarm', alpha=0.5)
plt.colorbar(sc, label='TARGET')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance expliquée)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance expliquée)")
plt.title("Projection PCA colorée par la cible")
plt.show()

In [ ]:
# --- A. Combien de composantes faut-il vraiment ? (scree plot + variance cumulée) ---
pca_full = PCA().fit(X_scaled)  # sans limiter à 2 composantes, pour voir toutes les infos

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
         pca_full.explained_variance_ratio_.cumsum(), marker='o')
plt.axhline(0.8, color='red', linestyle='--', label='Seuil 80% variance')
plt.xlabel("Nombre de composantes")
plt.ylabel("Variance cumulée expliquée")
plt.title("Scree plot - Variance cumulée par composante")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Variance expliquée par les 2 premières composantes :",
      f"{pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

In [ ]:
# --- B. Quelles variables construisent PC1, PC2, PC3 ? (loadings) ---
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2', 'PC3'],
    index=numerical_features
)

loadings_sorted = loadings.sort_values('PC1', key=abs, ascending=False)

print("--- Contribution des variables aux 3 premiers axes (loadings) ---")
print(loadings_sorted)

# Hauteur de figure proportionnelle au nombre de variables,
# pour garantir assez de place pour CHAQUE label
n_vars = len(loadings_sorted)
fig_height = max(8, n_vars * 0.3)  # 0.3 pouce par ligne, avec un minimum de 8

plt.figure(figsize=(6, fig_height))
ax = sns.heatmap(loadings_sorted,
                  cmap='coolwarm', center=0, annot=True, fmt=".2f",
                  yticklabels=1)  # <- force l'affichage de TOUS les labels, sans saut

ax.tick_params(axis='y', labelsize=8)  # ajuste la taille de police si besoin
plt.title("Contribution des variables à PC1 / PC2 / PC3")
plt.tight_layout()
plt.show()


In [ ]:
# ---  Tableau récapitulatif combiné (cos2 + contributions) ---
summary = pd.concat(
    [contrib_df.add_suffix('_contrib(%)'), cos2_df.add_suffix('_cos2')],
    axis=1
)

# Exemples de tris utiles (à adapter selon ce que tu veux visualiser) :
print("--- Trié par contribution à PC1 ---")
display(summary.sort_values('PC1_contrib(%)', ascending=False))

print("--- Trié par qualité de représentation globale (cos2 3D) ---")
display(summary.sort_values('cos2_total_3D_cos2', ascending=False))


In [ ]:
# Cercle de corrélation

from adjustText import adjust_text

def plot_correlation_circle(coord_df, cos2_df, axis_x, axis_y, cos2_col='cos2_total_3D'):
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_axes([0.05, 0.05, 0.70, 0.90])

    norm = plt.Normalize(cos2_df[cos2_col].min(), cos2_df[cos2_col].max())
    cmap = plt.cm.coolwarm

    texts = []  # on stocke les objets Text pour les passer à adjust_text ensuite

    for i, var in enumerate(coord_df.index):
        color = cmap(norm(cos2_df[cos2_col].iloc[i]))
        ax.arrow(0, 0, coord_df.loc[var, axis_x], coord_df.loc[var, axis_y],
                 head_width=0.02, color=color, alpha=0.8)
        # On crée le texte à la position de la flèche, mais on laisse adjust_text
        # décider ensuite de la position finale la plus lisible
        t = ax.text(coord_df.loc[var, axis_x], coord_df.loc[var, axis_y],
                     var, fontsize=7)
        texts.append(t)

    circle = plt.Circle((0, 0), 1, color='blue', fill=False)
    ax.add_artist(circle)
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel(f"{axis_x} ({pca.explained_variance_ratio_[int(axis_x[-1])-1]*100:.1f}%)")
    ax.set_ylabel(f"{axis_y} ({pca.explained_variance_ratio_[int(axis_y[-1])-1]*100:.1f}%)")
    ax.set_title(f"Cercle des corrélations - Plan {axis_x}/{axis_y}")
    ax.grid(alpha=0.3)

    # --- Répartition automatique des labels pour éviter les chevauchements ---
    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(arrowstyle='-', color='grey', lw=0.5, alpha=0.6),
        expand_text=(1.1, 1.2),
        force_text=(0.3, 0.3)
    )

    # --- Colorbar séparée, à droite ---
    cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, label=f"cos2 ({cos2_col})")

    plt.show()


# --- Plan factoriel 1 : PC1 / PC2 ---
plot_correlation_circle(coord_df, cos2_df, 'PC1', 'PC2')

# --- Plan factoriel 2 : PC1 / PC3 ---
plot_correlation_circle(coord_df, cos2_df, 'PC1', 'PC3')


In [ ]:
## ---------------------------------------------------------
## Détection des valeurs atypiques (outliers) : univariée + multivariée (PCA)
## Version optimisée mémoire / robustesse d'exécution
## ---------------------------------------------------------
import gc  # garbage collector, pour libérer la mémoire entre les étapes

## ---------------------------------------------------------
## 1. Détection univariée par IQR (vectorisée, sans boucle Python)
## ---------------------------------------------------------
def detect_outliers_iqr(df_input, cols, k=1.5):
    """
    Version vectorisée (numpy), plus rapide et moins gourmande en mémoire
    que la version avec boucle for classique.
    """
    q1 = df_input[cols].quantile(0.25)
    q3 = df_input[cols].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr

    # Comparaison vectorisée sur tout le DataFrame d'un coup
    outlier_flags = (df_input[cols] < lower) | (df_input[cols] > upper)
    return outlier_flags

iqr_flags = detect_outliers_iqr(df_pca_input, numerical_features)
nb_iqr_outliers_per_obs = iqr_flags.sum(axis=1)

print("--- Répartition du nombre de variables atypiques (IQR) par observation ---")
print(nb_iqr_outliers_per_obs.value_counts().sort_index())

# On limite l'affichage aux 15 variables les plus problématiques (au lieu de tout afficher)
print("\n--- Top 15 variables les plus souvent à l'origine d'outliers univariés ---")
print(iqr_flags.sum().sort_values(ascending=False).head(15))

## ---------------------------------------------------------
## 2. Détection multivariée via PCA (T² de Hotelling + statistique Q)
## ---------------------------------------------------------
from sklearn.decomposition import PCA
from scipy.stats import f as f_dist

# /!\ On calcule pca_full UNE SEULE FOIS (évite un doublon de calcul coûteux)
pca_full = PCA().fit(X_scaled)
n_comp_outliers = int(np.argmax(np.cumsum(pca_full.explained_variance_ratio_) >= 0.80) + 1)
print(f"\nNombre de composantes retenues pour la détection d'outliers : {n_comp_outliers}")

# On réutilise directement les scores déjà calculés par pca_full au lieu de refaire un fit
scores = pca_full.transform(X_scaled)[:, :n_comp_outliers]
eigenvalues = pca_full.explained_variance_[:n_comp_outliers]

T2 = np.sum((scores ** 2) / eigenvalues, axis=1)

n_obs = X_scaled.shape[0]
alpha = 0.05
T2_threshold = (n_comp_outliers * (n_obs - 1) / (n_obs - n_comp_outliers)) * \
               f_dist.ppf(1 - alpha, n_comp_outliers, n_obs - n_comp_outliers)

# Reconstruction limitée aux composantes retenues (pas besoin de reconstruire tout pca_full)
components_subset = pca_full.components_[:n_comp_outliers]
X_reconstructed = scores @ components_subset
Q = np.sum((X_scaled - X_reconstructed) ** 2, axis=1)
Q_threshold = np.percentile(Q, 95)

pca_outliers = pd.DataFrame({
    'T2': T2,
    'Q': Q,
    'outlier_T2': T2 > T2_threshold,
    'outlier_Q': Q > Q_threshold
}, index=df_pca_input.index)

del scores, X_reconstructed  # libère la mémoire des tableaux intermédiaires volumineux
gc.collect()

## ---------------------------------------------------------
## 3. Synthèse combinée : univarié + multivarié
## ---------------------------------------------------------
summary_outliers = pd.DataFrame(index=df_pca_input.index)
summary_outliers['nb_var_atypiques_IQR'] = nb_iqr_outliers_per_obs
summary_outliers['outlier_T2_PCA'] = pca_outliers['outlier_T2']
summary_outliers['outlier_Q_PCA'] = pca_outliers['outlier_Q']

summary_outliers['niveau_alerte'] = np.select(
    [
        (summary_outliers['nb_var_atypiques_IQR'] >= 3) & (summary_outliers['outlier_T2_PCA'] | summary_outliers['outlier_Q_PCA']),
        (summary_outliers['nb_var_atypiques_IQR'] >= 3) | summary_outliers['outlier_T2_PCA'] | summary_outliers['outlier_Q_PCA'],
        (summary_outliers['nb_var_atypiques_IQR'] > 0)
    ],
    ['Fort', 'Moyen', 'Faible'],
    default='Normal'
)

print("\n--- Répartition des niveaux d'alerte ---")
print(summary_outliers['niveau_alerte'].value_counts())

top_alertes = summary_outliers[summary_outliers['niveau_alerte'] == 'Fort'] \
    .sort_values('nb_var_atypiques_IQR', ascending=False)

print(f"\n--- {len(top_alertes)} observation(s) en alerte 'Fort' ---")

In [ ]:

import openpyxl
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter
import numpy as np

## ---------------------------------------------------------
## Export Excel : coloration par cellule selon sa sévérité (version vectorisée)
## ---------------------------------------------------------

output_path = SORTIE + 'outliers_detail.xlsx'
id_cols = [c for c in ['ID', 'MAC_CODE', 'Date_time'] if c in df.columns]

SEVERITY_THRESHOLDS = [
    ('Faible', 0, 1),
    ('Moyen',  1, 2),
    ('Fort',   2, np.inf),
]

COLOR_BY_SEVERITY = {
    'Faible': 'FFF2CC',
    'Moyen':  'F8CBAD',
    'Fort':   'FF6B6B',
}

q1 = df_pca_input[numerical_features].quantile(0.25)
q3 = df_pca_input[numerical_features].quantile(0.75)
iqr_s = q3 - q1
lower_bounds = q1 - 1.5 * iqr_s
upper_bounds = q3 + 1.5 * iqr_s

lower_arr = lower_bounds[numerical_features].values
upper_arr = upper_bounds[numerical_features].values
iqr_arr = iqr_s[numerical_features].values
iqr_arr_safe = np.where(iqr_arr == 0, np.nan, iqr_arr)

def compute_severity_labels(values):
    """
    Calcule, pour toute la matrice de valeurs d'un coup, un tableau de labels
    ('Faible'/'Moyen'/'Fort'/'') selon la distance normalisée à la borne IQR.
    """
    below = values < lower_arr
    above = values > upper_arr

    dist = np.zeros_like(values, dtype=float)
    iqr_broadcast = np.broadcast_to(iqr_arr_safe, values.shape)
    dist[below] = (lower_arr - values)[below] / iqr_broadcast[below]
    dist[above] = (values - upper_arr)[above] / iqr_broadcast[above]

    labels = np.full(values.shape, '', dtype=object)
    for label, low, high in SEVERITY_THRESHOLDS:
        mask = (dist >= low) & (dist < high) & (below | above)
        labels[mask] = label

    return labels

# --- 1. Sélection des observations atypiques ---
obs_index = summary_outliers[summary_outliers['niveau_alerte'] != 'Normal'].index

export_cols = id_cols + numerical_features
export_df = df.loc[obs_index, export_cols].copy()
export_df = export_df.join(summary_outliers.loc[obs_index, ['nb_var_atypiques_IQR',
                                                               'outlier_T2_PCA', 'outlier_Q_PCA',
                                                               'niveau_alerte']])
export_df['niveau_alerte'] = pd.Categorical(export_df['niveau_alerte'],
                                             categories=['Fort', 'Moyen', 'Faible'], ordered=True)
export_df = export_df.sort_values('niveau_alerte')

# --- 2. Calcul vectorisé de la sévérité pour TOUTE la matrice d'un coup ---
values = df.loc[export_df.index, numerical_features].values.astype(float)
severity_labels = compute_severity_labels(values)  # tableau 2D de labels

# --- 3. Écriture de la feuille ---
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    export_df.to_excel(writer, sheet_name='Outliers', index=False)

# --- 4. Coloration : on ne boucle QUE sur les cellules effectivement atypiques ---
wb = openpyxl.load_workbook(output_path)
ws = wb['Outliers']

col_offset = len(id_cols) + 1  # décalage pour atteindre les colonnes de numerical_features

rows_idx, cols_idx = np.where(severity_labels != '')
for r, c in zip(rows_idx, cols_idx):
    label = severity_labels[r, c]
    color = COLOR_BY_SEVERITY[label]
    ws.cell(row=r + 2, column=c + col_offset).fill = PatternFill(
        start_color=color, end_color=color, fill_type='solid'
    )

ws.freeze_panes = ws.cell(row=2, column=len(id_cols) + 1)
for i, col in enumerate(id_cols, start=1):
    ws.column_dimensions[get_column_letter(i)].width = 15

wb.save(output_path)
print(f"Fichier exporté : {output_path}")

# Vérification rapide de la répartition des sévérités
unique, counts = np.unique(severity_labels[severity_labels != ''], return_counts=True)
print("Répartition des cellules par sévérité :", dict(zip(unique, counts)))


In [ ]:
## ---------------------------------------------------------
## Visualisation croisée avec identification des points atypiques (version sécurisée)
## ---------------------------------------------------------
from adjustText import adjust_text

plot_data = summary_outliers.join(pca_outliers[['T2', 'Q']])

plt.figure(figsize=(10, 7))
ax = plt.gca()

sns.scatterplot(x='T2', y='Q', hue='niveau_alerte', data=plot_data,
                 palette={'Normal': 'grey', 'Faible': 'gold', 'Moyen': 'orange', 'Fort': 'red'},
                 alpha=0.6, ax=ax)

plt.axvline(T2_threshold, color='black', linestyle='--', linewidth=0.8)
plt.axhline(Q_threshold, color='black', linestyle=':', linewidth=0.8)
plt.title("Croisement détection univariée (IQR) et multivariée (PCA T²/Q)")
plt.xlabel("T² de Hotelling")
plt.ylabel("Q (erreur de reconstruction)")

# /!\ On ne labellise que les N cas les plus extrêmes, pas tous les "Fort"
# (1438 labels feraient planter adjust_text / le noyau)
N_LABELS_MAX = 30
to_label = plot_data[plot_data['niveau_alerte'] == 'Fort'] \
    .assign(score=lambda d: d['T2']/T2_threshold + d['Q']/Q_threshold) \
    .sort_values('score', ascending=False) \
    .head(N_LABELS_MAX)

texts = []
for idx, row in to_label.iterrows():
    label = df.loc[idx, 'ID'] if 'ID' in df.columns else str(idx)
    t = ax.text(row['T2'], row['Q'], label, fontsize=8, color='darkred')
    texts.append(t)

if texts:
    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(arrowstyle='-', color='darkred', lw=0.6, alpha=0.7),
        expand_text=(1.2, 1.3),
        force_text=(0.4, 0.4)
    )

plt.title(f"Croisement IQR / PCA (T²-Q) — top {N_LABELS_MAX} anomalies les plus extrêmes annotées")
plt.tight_layout()
plt.show()
